In [1]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("spam-experiment")

<Experiment: artifact_location='/mnt/c/Users/user/Documents/Projects/mlops-exer-1/notebooks/mlruns/1', creation_time=1780963326545, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780963326545, lifecycle_stage='active', name='spam-experiment', tags={}, trace_location=None, workspace='default'>

In [2]:
# from spam_checker.data.spam_data_module import SPAMDataModule

In [3]:
# spam_data = SPAMDataModule()

In [4]:
# spam_data.prepare_data()

In [5]:
import pandas as pd

# Paste your copied file path inside the quotes
# df = pd.read_csv('../test_dataset/spam.csv')

df = pd.read_csv('../test_dataset/spam.csv', encoding_errors='ignore')

df = df[['v1', 'v2']]

# df['v1'] = df['v1'].replace('spam', '1')
# df['v1'] = df['v1'].replace('ham', '0')
df = df.rename(columns={"v1": "label", "v2": "text"})

# View the first few rows of your data
df.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [6]:
# df['label'] = df['label'].replace('spam', '1')
# df['label'] = df['label'].replace('ham', '0')

# # View the first few rows of your data
# df.head(10)

In [7]:
len(df.index)

5572

In [8]:
empty_spam_rows = df[df['text'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [9]:
empty_count = df['text'].isna().sum()
print(f"Empty cells: {empty_count}")

Empty cells: 0


In [10]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.2 MB/s  0:00:01m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [11]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [12]:
doc = nlp("spaCy is successfully working in Jupyter!")
for token in doc:
    print(token.text, token.pos_)

spaCy NUM
is AUX
successfully ADV
working VERB
in ADP
Jupyter PROPN
! PUNCT


In [13]:
# Function to clean the text
def clean_text(text):
    doc = nlp(text)  # Process the text with spaCy
    cleaned_tokens = []
    for token in doc:
        # Remove stop words, punctuation, and retain only alphabetic words
        if not token.is_stop and not token.is_punct and token.is_alpha:
            cleaned_tokens.append(token.lemma_)  # Append the lemmatized version of the token
    return " ".join(cleaned_tokens)

# Apply the cleaning function to the 'text' column in the dataset
# df['cleaned_text'] = df['text'].apply(clean_text)

In [14]:
df.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [15]:
training_input_labels = df.loc[:, df.columns == 'label']
training_input_text = df.loc[:, df.columns != 'label']

print(training_input_labels['label'].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [16]:
print(training_input_labels)

     label
0      ham
1      ham
2     spam
3      ham
4      ham
...    ...
5567  spam
5568   ham
5569   ham
5570   ham
5571   ham

[5572 rows x 1 columns]


In [17]:
from sklearn.model_selection import train_test_split

train_text_temp, val_text_final, train_labels_temp, val_labels = train_test_split(
    training_input_text, training_input_labels, train_size=0.5, random_state=0, stratify=training_input_labels)

train_text_final, test_text_final, train_labels, test_labels = train_test_split(
    train_text_temp, train_labels_temp, train_size=0.8, random_state=0, stratify=train_labels_temp)


In [18]:
train_text_combined = pd.concat([train_labels, train_text_final], axis=1).reset_index(drop=True)

In [19]:
test_text_combined = pd.concat([test_labels, test_text_final], axis=1).reset_index(drop=True)

In [20]:
val_text_combined = pd.concat([val_labels, val_text_final], axis=1).reset_index(drop=True)

In [21]:
empty_spam_rows = train_text_combined[train_text_combined['text'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [22]:
# empty_spam_rows = train_text_combined[train_text_combined['cleaned_text'].isna()]
# print(empty_spam_rows)

In [23]:
empty_spam_rows = train_text_combined[train_text_combined['label'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [24]:
def spacy_tokenizer(text):
    # Use the clean_lemm function first
    cleaned_text = clean_text(text)
    # Then tokenize the cleaned text
    return cleaned_text.split()

In [25]:
from collections import defaultdict

def build_vocab(text_iterator, min_freq=3, specials=('<unk>', '<pad>')):
    token_counts = defaultdict(int)
    for text in text_iterator:
        for token in spacy_tokenizer(text):
            token_counts[token] += 1
    vocab = {token: idx for idx, (token, count) in enumerate(token_counts.items()) if count >= min_freq}
    for special in specials:
        if special not in vocab:
            vocab[special] = len(vocab) 
    return vocab
vocab = build_vocab(df['text'])
# vocab = build_vocab(df['cleaned_text'])
vocab_size = len(vocab)

In [26]:
print(vocab_size)

2216


In [27]:
print(vocab)

{'point': 1, 'crazy': 2, 'available': 3, 'bugis': 4, 'n': 5, 'great': 6, 'world': 7, 'la': 8, 'e': 9, 'get': 12, 'wat': 14, 'ok': 15, 'lar': 16, 'joke': 17, 'wif': 18, 'u': 19, 'oni': 20, 'free': 21, 'entry': 22, 'wkly': 23, 'comp': 24, 'win': 25, 'Cup': 27, 'final': 28, 'tkts': 29, 'text': 30, 'receive': 32, 'txt': 33, 'apply': 34, 'dun': 35, 'early': 36, 'c': 38, 'Nah': 39, 'think': 40, 'go': 41, 'usf': 42, 'live': 43, 'FreeMsg': 44, 'hey': 45, 'week': 47, 'word': 48, 'like': 49, 'fun': 50, 'tb': 51, 'std': 53, 'send': 55, 'brother': 57, 'speak': 58, 'treat': 59, 'request': 62, 'Melle': 63, 'Oru': 64, 'Minnaminunginte': 65, 'Nurungu': 66, 'Vettam': 67, 'set': 68, 'callertune': 69, 'Callers': 70, 'press': 71, 'copy': 72, 'friend': 73, 'Callertune': 74, 'WINNER': 75, 'value': 76, 'network': 77, 'customer': 78, 'select': 79, 'prize': 81, 'reward': 82, 'claim': 83, 'code': 84, 'valid': 85, 'hour': 86, 'mobile': 87, 'month': 88, 'U': 89, 'R': 90, 'entitle': 91, 'update': 92, 'late': 93, '

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# 1. Mock Data (Replace this with your SMS Spam Collection CSV file)
# data = [
#     ("Go jurong point, crazy.. Available only in bugis n great world la e buffet...", "ham"),
#     ("Ok lar... Joking wif u oni...", "ham"),
#     ("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.", "spam"),
#     ("U dun say so early hor... U c already then say...", "ham"),
#     ("FreeMsg Hey there darling it's been 3 week's now no word back!", "spam"),
#     ("WINNER!! As a valued network customer you have been selected to receivea £900 prize reward!", "spam"),
#     ("Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free!", "spam"),
#     ("I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k?", "ham"),
# ]

data = df
 

# 2. Text Preprocessing and Vocabulary Builder
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

# Build Vocabulary
all_words = []
for text in data['text']:
    all_words.extend(clean_text(text))

word_counts = Counter(all_words)
# Filter words that appear at least once; add <PAD> and <UNK> tokens
vocab = {"<PAD>": 0, "<UNK>": 1}
for word in word_counts:
    if word not in vocab:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
MAX_LEN = 20  # Max sequence length for padding

print(vocab)

{'<PAD>': 0, '<UNK>': 1, 'go': 2, 'until': 3, 'jurong': 4, 'point': 5, 'crazy': 6, 'available': 7, 'only': 8, 'in': 9, 'bugis': 10, 'n': 11, 'great': 12, 'world': 13, 'la': 14, 'e': 15, 'buffet': 16, 'cine': 17, 'there': 18, 'got': 19, 'amore': 20, 'wat': 21, 'ok': 22, 'lar': 23, 'joking': 24, 'wif': 25, 'u': 26, 'oni': 27, 'free': 28, 'entry': 29, '2': 30, 'a': 31, 'wkly': 32, 'comp': 33, 'to': 34, 'win': 35, 'fa': 36, 'cup': 37, 'final': 38, 'tkts': 39, '21st': 40, 'may': 41, '2005': 42, 'text': 43, '87121': 44, 'receive': 45, 'questionstd': 46, 'txt': 47, 'ratetcs': 48, 'apply': 49, '08452810075over18s': 50, 'dun': 51, 'say': 52, 'so': 53, 'early': 54, 'hor': 55, 'c': 56, 'already': 57, 'then': 58, 'nah': 59, 'i': 60, 'dont': 61, 'think': 62, 'he': 63, 'goes': 64, 'usf': 65, 'lives': 66, 'around': 67, 'here': 68, 'though': 69, 'freemsg': 70, 'hey': 71, 'darling': 72, 'its': 73, 'been': 74, '3': 75, 'weeks': 76, 'now': 77, 'and': 78, 'no': 79, 'word': 80, 'back': 81, 'id': 82, 'like'

In [30]:
# 3. Custom PyTorch Dataset
class SMSDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data['text']
        self.labels = data['label']
        self.vocab = vocab
        self.max_len = max_len
        self.labels_map = {"ham": 0, "spam": 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]
        label = self.labels[idx]
        words = clean_text(text)
        
        # Numericalize text (Word -> ID)
        encoded = [self.vocab.get(word, self.vocab["<UNK>"]) for word in words]
        
        # Pad or truncate sequence to fixed length
        if len(encoded) < self.max_len:
            encoded += [self.vocab["<PAD>"]] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]
            
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels_map[label], dtype=torch.float)

# Create Dataset and DataLoader
dataset = SMSDataset(data, vocab, MAX_LEN)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)



In [31]:
# 4. Simple RNN Classification Model
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SpamClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: [batch_size, max_len]
        embedded = self.embedding(x) # [batch_size, max_len, embedding_dim]
        lstm_out, (hidden, cell) = self.lstm(embedded) # hidden shape: [1, batch_size, hidden_dim]
        
        # Use the final hidden state of the LSTM
        last_hidden = hidden.squeeze(0) # [batch_size, hidden_dim]
        out = self.fc(last_hidden) # [batch_size, 1]
        return self.sigmoid(out)

# Hyperparameters
EMBEDDING_DIM = 32
HIDDEN_DIM = 16

model = SpamClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 5. Training Loop
print("Starting Training...")
model.train()
for epoch in range(5):
    epoch_loss = 0
    for texts, labels in dataloader:
        optimizer.zero_grad()
        
        predictions = model(texts).squeeze(1) # shape matches labels [batch_size]
        loss = criterion(predictions, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(dataloader):.4f}")

# 6. Inference Example
def predict(text):
    model.eval()
    with torch.no_grad():
        words = clean_text(text)
        encoded = [vocab.get(word, vocab["<UNK>"]) for word in words]
        if len(encoded) < MAX_LEN:
            encoded += [vocab["<PAD>"]] * (MAX_LEN - len(encoded))
        else:
            encoded = encoded[:MAX_LEN]
            
        test_tensor = torch.tensor([encoded], dtype=torch.long)
        prediction = model(test_tensor).item()
        return "Spam" if prediction > 0.5 else "Ham"

print("\nTesting Predictions:")
print(f"'Free entry to win cash now!' -> {predict('Free entry to win cash now!')}")
print(f"'Hey, are we still meeting for lunch?' -> {predict('Hey, are we still meeting for lunch?')}")

Starting Training...
Epoch 1 | Loss: 0.1690
Epoch 2 | Loss: 0.0685
Epoch 3 | Loss: 0.0317
Epoch 4 | Loss: 0.0181
Epoch 5 | Loss: 0.0167

Testing Predictions:
'Free entry to win cash now!' -> Spam
'Hey, are we still meeting for lunch?' -> Ham
